# M6a · Build `marts.mart_device_monthly_behavior`

**Goal:** run `sql/20_mart_device_monthly_behavior.sql` for every month that got data during the fact backfill.

After a clean backfill we don't yet know the exact list of touched months, so we pass the full distinct month list computed from `warehouse.fact_trip`.

**Exit criterion:** `SELECT COUNT(*) FROM marts.mart_device_monthly_behavior > 0` and the 35 feature columns have plausible distributions.


In [1]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
# Make the src/ package importable even if pip install -e was skipped.
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


DB = medali_dev@localhost:15432/accent_data
Schemas: staging=staging, warehouse=warehouse, marts=marts


## 2. Inputs

In [2]:
import pandas as pd
with get_engine().connect() as conn:
    months = pd.read_sql(text("""
        SELECT DISTINCT TO_CHAR(begin_path_time, 'YYYY-MM') AS ym
        FROM warehouse.fact_trip ORDER BY ym
    """), conn)['ym'].tolist()
print(f"{len(months)} distinct months to recompute")
print(months[:12], '...' if len(months) > 12 else '')


79 distinct months to recompute
['2019-10', '2019-11', '2019-12', '2020-01', '2020-02', '2020-03', '2020-04', '2020-05', '2020-06', '2020-07', '2020-08', '2020-09'] ...


## 3. Execute

In [3]:
import importlib
import time
import accent_fleet.db as db_module
import accent_fleet.db.sql_loader as sql_loader_module
from accent_fleet.pipeline.run_log import begin_run, end_run

# A notebook kernel can keep old imported function objects alive after source edits.
importlib.reload(sql_loader_module)
db_module = importlib.reload(db_module)
run_sql_file = db_module.run_sql_file
transaction = db_module.transaction

run_id = begin_run(mode='notebook:mart_device_monthly_behavior')
t0 = time.time()
try:
    with transaction() as conn:
        result = run_sql_file(conn, '20_mart_device_monthly_behavior.sql',
                              params={'touched_months': months, 'etl_run_id': run_id})
    rows = result.rowcount or 0
    end_run(run_id, status='success', rows_loaded=rows)
    print(f"mart recomputed: {rows} rows in {time.time()-t0:.1f}s")
except Exception as exc:
    end_run(run_id, status='failed', error_message=str(exc))
    raise


mart recomputed: 23965 rows in 216.4s


## 4. Inspect

In [4]:
import pandas as pd
with get_engine().connect() as conn:
    total = conn.execute(text("SELECT COUNT(*) FROM marts.mart_device_monthly_behavior")).scalar_one()
    sample = pd.read_sql(text("SELECT * FROM marts.mart_device_monthly_behavior ORDER BY year_month DESC LIMIT 10"), conn)
    stats = pd.read_sql(text("""
      SELECT
        AVG(total_trips) avg_trips,
        AVG(total_distance_km) avg_distance,
        AVG(overspeed_count) avg_overspeed,
        AVG(night_trip_ratio) avg_night_ratio
      FROM marts.mart_device_monthly_behavior
    """), conn)
print(f"mart rows: {total:,}")
display(stats); display(sample)
assert total > 0, 'mart is empty'


mart rows: 23,965


,avg_trips,avg_distance,avg_overspeed,avg_night_ratio
0,307.435969,4406.405197,37.575005,0.114251


,tenant_id,device_id,year_month,total_trips,total_distance_km,avg_trip_distance_km,avg_trip_duration_minutes,avg_fuel_used_l,stddev_trip_distance,short_trip_ratio,...,short_stop_count,medium_stop_count,long_stop_count,night_trip_ratio,weekend_trip_ratio,rush_hour_trip_ratio,active_days,avg_working_hours,_etl_run_id,_computed_at
0,1787,8450,2026-04,41,147.495965,3.597463,45.408943,0.000000,2.847533,0.243902,...,22,7,8,0.024390,0.756098,0.121951,5,21.083056,34,2026-04-29 23:01:34.576127+00:00
1,235,425318,2026-04,123,3032.352745,24.653274,31.771816,0.000000,42.887896,0.422764,...,69,30,23,0.260163,0.089431,0.162602,10,11.655093,34,2026-04-29 23:01:34.576127+00:00
2,235,425206,2026-04,166,1291.095176,7.777682,14.987751,0.000000,18.809649,0.554217,...,102,35,28,0.102410,0.066265,0.216867,10,7.459259,34,2026-04-29 23:01:34.576127+00:00
3,235,425226,2026-04,61,1125.208585,18.446042,23.997814,6.683333,27.050484,0.196721,...,31,7,21,0.213115,0.114754,0.114754,9,4.461914,34,2026-04-29 23:01:34.576127+00:00
4,235,425217,2026-04,127,2988.295791,23.529888,35.342520,0.000000,29.544099,0.228346,...,78,29,19,0.165354,0.141732,0.196850,9,11.070913,34,2026-04-29 23:01:34.576127+00:00
5,235,425177,2026-04,74,882.250392,11.922303,21.717568,4.246575,18.539655,0.459459,...,57,9,6,0.000000,0.175676,0.324324,7,5.051944,34,2026-04-29 23:01:34.576127+00:00
6,264,436322,2026-04,142,4491.100069,31.627465,35.529108,0.000000,34.007956,0.077465,...,58,51,32,0.260563,0.190141,0.239437,10,NaN,34,2026-04-29 23:01:34.576127+00:00
7,1787,8460,2026-04,193,677.993436,3.512919,8.007254,0.000000,6.491488,0.430052,...,141,38,12,0.000000,0.145078,0.253886,8,4.215903,34,2026-04-29 23:01:34.576127+00:00
8,238,427680,2026-04,102,2897.019844,28.402155,36.299673,0.000000,46.655857,0.245098,...,51,26,23,0.147059,0.000000,0.205882,8,8.099028,34,2026-04-29 23:01:34.576127+00:00
9,235,425139,2026-04,83,762.453619,9.186188,17.497791,0.000000,9.381978,0.192771,...,50,23,8,0.000000,0.144578,0.240964,9,3.837153,34,2026-04-29 23:01:34.576127+00:00
